In [2]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Patel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

STOP_WORDS = set(stopwords.words("english")) - {"not", "no", "nor"}
stemmer = PorterStemmer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = re.findall(r"\b\w+\b", text)
    return " ".join(stemmer.stem(t) for t in tokens if t not in STOP_WORDS)

In [4]:
data = [
    ("driver has not arrived yet", "Delivery"),
    ("order is two hours late", "Delivery"),
    ("my food was never delivered", "Delivery"),
    ("delivery person went to wrong address", "Delivery"),
    ("tracking says arriving but no driver", "Delivery"),
    ("package arrived much later than promised", "Delivery"),
    ("courier cancelled my order", "Delivery"),
    ("missing delivery and no update", "Delivery"),
    ("driver is late and app tracking is broken", "Delivery"),
    ("delivery was delayed again today", "Delivery"),
    ("pizza arrived cold and hard", "Food Quality"),
    ("burger was raw in the middle", "Food Quality"),
    ("soup spilled inside the bag", "Food Quality"),
    ("fries were soggy and stale", "Food Quality"),
    ("food tasted burnt and salty", "Food Quality"),
    ("wrong dish was packed", "Food Quality"),
    ("salad was not fresh", "Food Quality"),
    ("drink leaked all over my meal", "Food Quality"),
    ("cold noodles and poor food quality", "Food Quality"),
    ("chicken smelled bad and tasted sour", "Food Quality"),
    ("app crashes when I open it", "App"),
    ("payment failed in the application", "App"),
    ("cannot log into my account", "App"),
    ("checkout button does not work", "App"),
    ("the order screen is frozen", "App"),
    ("promo code is not accepted by app", "App"),
    ("unable to track order in app", "App"),
    ("application keeps logging me out", "App"),
    ("payment page freezes during checkout", "App"),
    ("app will not load my saved address", "App"),
]

texts = [preprocess_text(text) for text, label in data]
labels = [label for text, label in data]
print("Total examples:", len(texts))
print("Class counts:")
print(pd.Series(labels).value_counts() if False else {label: labels.count(label) for label in sorted(set(labels))})

Total examples: 30
Class counts:
{'App': 10, 'Delivery': 10, 'Food Quality': 10}


In [6]:
# 2

X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts, labels, test_size=0.20, random_state=42, stratify=labels
)

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

model = MultinomialNB()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Classification report:")
print(classification_report(y_test, predictions, zero_division=0))

label_order = ["App", "Delivery", "Food Quality"]
print("Confusion matrix (rows=true, columns=predicted)")
print("Label order:", label_order)
print(confusion_matrix(y_test, predictions, labels=label_order))

Classification report:
              precision    recall  f1-score   support

         App       0.33      0.50      0.40         2
    Delivery       0.00      0.00      0.00         2
Food Quality       0.50      0.50      0.50         2

    accuracy                           0.33         6
   macro avg       0.28      0.33      0.30         6
weighted avg       0.28      0.33      0.30         6

Confusion matrix (rows=true, columns=predicted)
Label order: ['App', 'Delivery', 'Food Quality']
[[1 1 0]
 [1 0 1]
 [1 0 1]]


In [8]:
# 3

new_complaint = input("Enter a customer complaint: ")
new_vector = vectorizer.transform([preprocess_text(new_complaint)])
print("Predicted category:", model.predict(new_vector)[0])

Enter a customer complaint:  nothing


Predicted category: App
